In [ ]:
!pip install -qU llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai arxiv pypdf langchain-text-splitters chromadb llama-index-vector-stores-chroma "opentelemetry-api<=1.42.1" "opentelemetry-sdk<=1.42.1"

In [ ]:
!pip install -qU langchain-google-genai

In [ ]:
!pip install -qU nest_asyncio

import nest_asyncio
nest_asyncio.apply()

In [ ]:
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"]=userdata.get("GEMINI_API_KEY")

import requests,io,arxiv,chromadb
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from llama_index.core import Document,Settings,VectorStoreIndex,StorageContext
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore

Settings.llm=GoogleGenAI(model="gemini-3.5-flash")
Settings.embed_model=GoogleGenAIEmbedding(model="gemini-embedding-001")

In [ ]:
import time, json, re
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

current_paper_title = None
current_index = None
current_num_chunks = 0
current_quiz = []

#Tool for loading the paper from the platform arXiv and applying the rag pipeline on the loaded pdf for further usage.
@tool
def load_paper(query: str) -> str:
    """Search arXiv for a paper by title or topic, fetch its full PDF, and index it for question answering. Call this first before asking questions about the paper."""
    global current_index, current_paper_title, current_num_chunks, current_quiz
    try:
        client = arxiv.Client()
        search = arxiv.Search(query=f'ti:"{query}"', max_results=1)
        result = next(client.results(search))

        pdf_response = requests.get(result.pdf_url, timeout=30)
        reader = PdfReader(io.BytesIO(pdf_response.content))
        paper_text = "\n".join(page.extract_text() for page in reader.pages)

        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        chunks = splitter.split_text(paper_text)
        docs = [Document(text=chunk) for chunk in chunks]

        chroma_client = chromadb.PersistentClient(path="./chroma_papers")
        collection_name = "current_paper"
        try:
            chroma_client.delete_collection(collection_name)
        except Exception:
            pass
        chroma_collection = chroma_client.get_or_create_collection(collection_name)
        vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
        storage_context = StorageContext.from_defaults(vector_store=vector_store)
        index = VectorStoreIndex.from_documents(docs, storage_context=storage_context)

        current_index = index
        current_paper_title = result.title
        current_num_chunks = len(chunks)
        current_quiz = []

        time.sleep(3)
        return f"Loaded '{result.title}' ({len(chunks)} chunks indexed). You can now ask questions about the paper."
    except StopIteration:
        return f"No arXiv paper found with the title '{query}'. Ask the user to check the exact title."
    except Exception as e:
        return f"Failed to load the paper: {e}. Do not answer questions about the paper using general knowledge. Report the failure to the user instead."

#Tool to communicate and ask questions regarding the paper.
@tool
def ask_paper(question: str) -> str:
    """Answer a specific question about the currently loaded paper, using retrieved chunks from its full text."""
    if current_index is None:
        return "No paper is loaded yet. Call load_paper first. Do not answer from general knowledge."
    try:
        engine = current_index.as_query_engine(similarity_top_k=3)
        return str(engine.query(question))
    except Exception as e:
        return f"Question answering failed: {e}. Report this error to the user. Do not answer from general knowledge."

#Tool for summarization of the entire paper.
@tool
def summarize_paper() -> str:
    """Generate a comprehensive summary of the currently loaded paper: its main contribution, methods, and key findings."""
    if current_index is None:
        return "No paper is loaded yet. Call load_paper first. Do not summarize from general knowledge."
    try:
        engine = current_index.as_query_engine(
            response_mode="tree_summarize",
            similarity_top_k=current_num_chunks,
        )
        return str(engine.query(
            "Provide a comprehensive summary of this paper: its main contribution, methods, and key findings."
        ))
    except Exception as e:
        return f"Failed to summarize the paper: {e}. Report this error to the user. Do not summarize from general knowledge."

#Tool for generating MCQs and evaluating the answers given by user to test their knowledge on the paper.
@tool
def quiz_paper(num_questions: int = 5, answers: list[str] | None = None) -> str:
    """Quiz the user on the currently loaded paper. Call WITHOUT answers to generate new questions.
    Call WITH answers (letters in question order, e.g. ["B", "C", "A"]) to grade the user's replies to the current quiz."""
    global current_quiz
    if current_index is None:
        return "No paper is loaded yet. Call load_paper first. Do not generate questions from general knowledge."

    # checking the answers given by user
    if answers:
        if not current_quiz:
            return "There is no active quiz. Generate one first."
        if len(answers) != len(current_quiz):
            return f"The quiz has {len(current_quiz)} questions but {len(answers)} answers were given. Ask the user to answer all of them."
        score, report = 0, []
        for i, (q, a) in enumerate(zip(current_quiz, answers), 1):
            correct = q["answer"].strip().upper()
            given = a.strip().upper()
            if given == correct:
                score += 1
                report.append(f"Question {i}: Correct ({correct}). {q['explanation']}")
            else:
                report.append(f"Question {i}: Incorrect. You chose {given}, the correct answer is {correct}. {q['explanation']}")
        return f"Score: {score}/{len(current_quiz)}\n" + "\n".join(report)

    # Generating the quiz questions for the user when asked
    try:
        engine = current_index.as_query_engine(
            response_mode="tree_summarize",
            similarity_top_k=min(15, current_num_chunks),
        )
        prompt = (
            f"Generate {num_questions} multiple-choice questions testing understanding of this paper's key concepts. "
            "Respond with ONLY a valid JSON list, no extra text and no code fences. "
            'Each item must look like: {"question": "...", "options": {"A": "...", "B": "...", "C": "...", "D": "..."}, '
            '"answer": "B", "explanation": "one line"}'
        )
        raw = str(engine.query(prompt)).strip()
        raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()
        current_quiz = json.loads(raw)

        required = {"question", "options", "answer", "explanation"}
        if not isinstance(current_quiz, list) or not all(isinstance(q, dict) and required <= q.keys() for q in current_quiz):
            raise ValueError("The model returned a quiz with missing fields")

        lines = []
        for i, q in enumerate(current_quiz, 1):
            lines.append(f"Question {i}: {q['question']}")
            for letter, text in q["options"].items():
                lines.append(f"  {letter}) {text}")
            lines.append("")
        return "\n".join(lines) + "\nAsk the user to reply with their answers, e.g. 1-A, 2-C, 3-B. Do not reveal answers."
    except Exception as e:
        current_quiz = []
        return f"Failed to generate MCQs: {e}. Report this error to the user. Do not generate questions from general knowledge."


tools = [load_paper, ask_paper, summarize_paper, quiz_paper]

agent = create_agent(
    model="google_genai:gemini-3.5-flash",
    tools=tools,
    system_prompt=(
        "You are a research assistant for academic papers. "
        "When the user asks to load a paper, call ONLY load_paper and then stop, "
        "report the result, and ask what they'd like to do next (summary, a question, or a quiz). "
        "For specific questions use ask_paper. For an overview use summarize_paper. "
        "If the user wants a quiz or MCQs, call quiz_paper without answers. "
        "After that, show the questions and ask the user to reply with their answers. Never reveal or guess the answers yourself. "
        "When the user replies with answers, call quiz_paper with the answer letters in order and report its result exactly. "
        "CRITICAL: If any tool returns a failure or 'no paper loaded' message, report that exact message honestly. "
        "Never answer from your own training knowledge as a substitute, and never invent a reason for a tool failure. "
        "Remember context across the conversation so the user doesn't need to repeat the paper name."
    ),
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "research-session-test"}}

In [ ]:
from IPython.display import Markdown, display


def get_text(response):
    """Return only the plain text of the agent's final reply."""
    content = response["messages"][-1].content
    if isinstance(content, list):
        content = "".join(part.get("text", "") for part in content if isinstance(part, dict))
    return content

def run_chat():
    """Interactive chat with the research assistant. Type 'exit' to stop."""
    print("Your Research assistant ready.\n"
        "Tell me which paper to load (use the exact arXiv title), then ask questions, "
        "request a summary, or take a quiz to test your knowledge.\n"
        "Type 'exit' to stop.\n"
        )
    while True:
        user_input = input("You: ").strip()
        if not user_input:
            continue
        if user_input.lower() in {"exit", "quit"}:
            print("Chat ended.")
            break
        try:
            response = agent.invoke(
                {"messages": [{"role": "user", "content": user_input}]},
                config=config,
            )
            display(Markdown(f"**Assistant:**\n\n{get_text(response)}"))
        except Exception as e:
            print(f"Error: {e}\n")


run_chat()

Your Research assistant ready.
Tell me which paper to load (use the exact arXiv title), then ask questions, request a summary, or take a quiz to test your knowledge.
Type 'exit' to stop.

You: Can you summarize the paper Probalistic Explanations for Linear Models.


**Assistant:**

Here is a comprehensive summary of the paper **"Probabilistic Explanations for Linear Models"**:

---

### **Main Contribution**
Computing small probabilistic sufficient reasons ($\delta$-SRs)—subsets of input features that guarantee the classifier's outcome with a probability of at least $\delta$—is generally intractable, even for simple models like decision trees. This paper introduces **$(\delta, \epsilon)$-min-SRs**, a relaxation of $\delta$-SRs where the target confidence parameter $\delta$ is allowed to vary within a small interval $[\delta - \epsilon, \delta + \epsilon]$. 

The central contribution is proving that **$(\delta, \epsilon)$-min-SRs can be computed efficiently in polynomial time for linear models**, making linear models the first model class with guaranteed theoretical efficiency for such probabilistic explanations. The paper also provides structural insights showing that for linear models under product distributions, locally minimal $\delta$-SRs are guaranteed to be globally subset-minimal.

---

### **Methods**
The efficient computation of $(\delta, \epsilon)$-min-SRs relies on three key technical components:

1. **Greedy Feature Selection via Feature Scoring:**  
   For a linear classifier $\mathcal{L} = (\mathbf{w}, t)$ and an input $\mathbf{x}$, each feature $i$ is assigned a score:
   $$s_i := w_i \cdot (2x_i - 1) \cdot (2\mathcal{L}(\mathbf{x}) - 1)$$
   A positive score means the feature supports the classification outcome, whereas a negative score hurts it. Sorting features in descending order of their scores yields an optimal ordering for greedy feature selection under product distributions.

2. **Randomized Smoothing of the Target Probability $\delta$:**  
   Exact determination of minimum $\delta$-SR sizes is computationally hard due to worst-case threshold values of $\delta$. To bypass this, the algorithm samples a candidate target probability $\delta^*$ uniformly at random from $[\delta - \epsilon, \delta + \epsilon]$, avoiding exact-threshold bottlenecks.

3. **Monte Carlo Estimation & Binary Search:**  
   Because exact model evaluation over partial instances is #P-hard, acceptance probabilities are estimated using Monte Carlo sampling. Combining these estimates with binary search over the score-sorted features finds the minimal feature set required to achieve the target probability with high confidence ($1 - \gamma$).

---

### **Key Findings**

* **Polynomial Time Complexity:**  
  A $(\delta, \epsilon)$-min-SR can be computed with probability at least $1 - \gamma$ in polynomial time:
  $$\tilde{O}\left(\frac{d}{\epsilon^2 \gamma^2}\right)$$
  where $d$ is the number of features.

* **Equivalence of Local and Subset Minimality:**  
  Under any product distribution (independent features), every locally minimal $\delta$-SR (where removing any single feature drops the probability below $\delta$) is also globally subset-minimal for linear models.

* **#P-hardness of Exact Computation:**  
  Calculating the *exact* size of the smallest $\delta$-SR for linear models cannot be done in polynomial time unless $\text{FP} = \text{\#P}$, justifying the necessity of the $(\delta, \epsilon)$ relaxation.

* **Explanation Size Instability:**  
  Approximating $\delta$ within an $\epsilon$-interval does not guarantee an approximation of the explanation size itself. There exist linear models where increasing $\delta$ to $\delta + \epsilon$ causes the required explanation size to jump by $\Omega(d^{1/2 - \gamma})$.

---

Would you like to ask specific questions about the paper or take a quiz to test your understanding?

You: What is the method greedy feature selection?


**Assistant:**

In the context of this paper, **Greedy Feature Selection** is a method used to efficiently construct small probabilistic sufficient reasons ($\delta$-SRs) for linear models.

---

### **1. How It Works**

1. **Feature Scoring:**  
   For a linear classifier $\mathcal{L} = (\mathbf{w}, t)$ and a given input $\mathbf{x}$, each feature $i$ is assigned a score $s_i$:
   $$s_i = w_i \cdot (2x_i - 1) \cdot (2\mathcal{L}(\mathbf{x}) - 1)$$
   * **Sign:** The factor $(2x_i - 1) \cdot (2\mathcal{L}(\mathbf{x}) - 1)$ checks whether feature $i$ supports ($+1$) or opposes ($-1$) the classifier's output decision.
   * **Magnitude:** The magnitude of $s_i$ is proportional to the feature's weight $|w_i|$.

2. **Sorting:**  
   Features are sorted in descending order according to their scores $s_i$. Features with higher positive scores provide the strongest support for maintaining the model's classification decision.

3. **Greedy Selection:**  
   Features are selected one by one in this sorted order until the probability of retaining the classifier's output reaches or exceeds the target confidence threshold $\delta$.

---

### **2. Why It Is Used**

* **Avoids Exhaustive Search:**  
  For general model classes (such as decision trees), finding a minimal-sized probabilistic explanation requires searching through an exponential number of feature combinations, which is computationally intractable.
* **Optimality Under Product Distributions:**  
  For linear models under product/uniform distributions (where features are independent), unselected features contribute to the classifier's linear sum purely based on their weights and probability distributions. Because of this linear independence, selecting features strictly by highest score $s_i$ yields the optimal subset of features to maximize confidence for any given size, eliminating the need for exhaustive subset exploration.

---

Feel free to ask further questions about the paper or request a quiz!

You: Give me a quiz with 3 questions on this paper.


**Assistant:**

Here is a 3-question quiz on the paper:

---

### **Question 1**
What is the computational complexity of computing a minimum $1$-sufficient reason ($1$-SR) for linear models?  
- **A)** NP-hard  
- **B)** $\Sigma_2^p$-hard  
- **C)** Polynomial-time solvable  
- **D)** #P-hard  

---

### **Question 2**
How does the proposed algorithm select $\delta^*$ to maintain computational tractability when computing probabilistic explanations?  
- **A)** It sets $\delta^*$ to the maximum confidence parameter $\delta + \epsilon$.  
- **B)** It samples $\delta^*$ uniformly at random from the range $[\delta - \epsilon, \delta + \epsilon]$.  
- **C)** It selects $\delta^*$ deterministically based on feature weights.  
- **D)** It estimates $\delta^*$ by enumerating all feature subsets.  

---

### **Question 3**
What is the computational complexity of exactly computing the classification probability $\Pr_{\mathbf{z} \sim \mathcal{U}(\mathbf{y})}[\mathcal{L}(\mathbf{z}) = \mathcal{L}(\mathbf{x})]$ for linear models?  
- **A)** Polynomial-time solvable  
- **B)** NP-complete  
- **C)** #P-hard  
- **D)** PSPACE-hard  

---

Please reply with your answers (e.g., **1-A, 2-B, 3-C** or just **A, B, C**), and I will grade them for you!

You: 1- A,2-B,3-A


**Assistant:**

Here are the results of your quiz:

**Score:** 1/3

* **Question 1:** **Incorrect**  
  * **Your answer:** A  
  * **Correct answer:** C (Polynomial-time solvable)  
  * *Explanation:* Computing a minimum $1$-sufficient reason ($1$-SR) is polynomial-time solvable for linear models.

* **Question 2:** **Correct** (B)  
  * *Explanation:* The algorithm samples $\delta^*$ uniformly at random from $[\delta - \epsilon, \delta + \epsilon]$ to avoid computationally hard edge cases.

* **Question 3:** **Incorrect**  
  * **Your answer:** A  
  * **Correct answer:** C (#P-hard)  
  * *Explanation:* Computing the exact classification probability under uniform distribution for linear models is #P-hard (which is why Monte Carlo estimation is used instead).

You: Exit
Chat ended.


In [ ]:
# check what are tools called by the agents for the current config.
for m in agent.get_state(config).values["messages"]:
    for call in (getattr(m, "tool_calls", None) or []):
        print(f"{call['name']}  {call['args']}")

load_paper  {'query': 'Probabilistic Explanations for Linear Models'}
summarize_paper  {}
ask_paper  {'question': 'What is greedy feature selection in the context of this paper? How does it work and why is it used?'}
quiz_paper  {'num_questions': 3}
quiz_paper  {'answers': ['A', 'B', 'A']}
